# Task 3 | Exception queue and Finance sign-off

Run notebooks 01–04 for the same release and both cutoffs before running this notebook.
Use the catalog, schema and target payment day used by notebook 04. This notebook
publishes the exception queue and daily close for both cutoffs, cutoff changes and
the seven-day investigation at C2. All amounts are in integer minor units.

The allocation of payment amounts follows Q2. A PENDING case remains in the queue
as a monitoring item. Each blocking reason counts as one case. Source-only cases
with a known date are assigned to their reported bank and currency where available.
Missing bank or currency is shown as UNKNOWN. Cases with no reliable attribution date
stay in the queue with a null date and appear in a separate result for Finance review.


In [0]:
from datetime import date, timedelta
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "bdcc_g_dev")
dbutils.widgets.text("payment_day", "2026-01-31")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
day = date.fromisoformat(dbutils.widgets.get("payment_day").strip())
window_start = day - timedelta(days=6)

for name in (catalog, schema):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*", name), name

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
spark.sql("SET TIME ZONE 'UTC'")

def publish(name, query):
    spark.sql(f"CREATE OR REPLACE TABLE {name} USING DELTA AS\n{query}")


## Reconstruct the window at the two reporting cutoffs

This uses the financial SQL from notebook 04 at commit 1e777ec with its single-day
filter expanded to seven days. Each historical payment is evaluated at the target
day's C1 and C2 timestamps. The D slice is checked against the published Q2 close.


In [0]:
spark.sql(f"""CREATE OR REPLACE TEMP VIEW q3_window AS
WITH
pay AS (
  SELECT g.cutoff_id, g.cutoff_ts, to_date(g.request_timestamp) AS payment_day, g.gateway_system, g.gateway_reference, g.business_key AS payment_key,
         g.currency_cd AS requested_currency, coalesce(t.acquiring_bank_id, 'UNKNOWN') AS bank_id
  FROM clean_gateway_requests g
  LEFT JOIN clean_terminal_assignments t
    ON t.cutoff_id = g.cutoff_id AND t.terminal_system = g.terminal_system AND t.terminal_id = g.terminal_id
   AND g.request_timestamp >= t.valid_from AND g.request_timestamp < t.valid_to
  WHERE to_date(g.request_timestamp) BETWEEN DATE'{window_start}' AND DATE'{day}'
),
map AS (
  SELECT DISTINCT cutoff_id, from_system, from_reference, gateway_system, gateway_reference
  FROM clean_reference_mappings
),
pe AS (
  SELECT p.*, m.gateway_system, m.gateway_reference
  FROM clean_processor_events p
  JOIN map m ON m.cutoff_id = p.cutoff_id AND m.from_system = p.processor_system
            AND m.from_reference = p.processor_reference
),
caps AS (SELECT * FROM pe WHERE event_type = 'CAPTURE'),
movements AS (
  SELECT cutoff_id, gateway_system, gateway_reference, currency_cd, amount_minor AS amt FROM caps
  UNION ALL
  SELECT r.cutoff_id, r.gateway_system, r.gateway_reference, r.currency_cd, -r.amount_minor
  FROM pe r
  JOIN caps c ON c.cutoff_id = r.cutoff_id AND c.processor_system = r.processor_system
             AND c.event_id = r.related_event_id
  WHERE r.event_type = 'REVERSAL'
  UNION ALL
  SELECT f.cutoff_id, m.gateway_system, m.gateway_reference, f.currency_cd, -f.amount_minor
  FROM clean_refund_events f
  JOIN map m ON m.cutoff_id = f.cutoff_id AND m.from_system = f.processor_system
            AND m.from_reference = f.processor_reference
  WHERE f.refund_status = 'COMPLETED'
),
expected AS (
  SELECT cutoff_id, gateway_system, gateway_reference, currency_cd, sum(amt) AS expected_minor
  FROM movements GROUP BY ALL
),
due AS (
  SELECT cutoff_id, gateway_system, gateway_reference, max(settlement_due_at) AS settlement_due_at
  FROM caps GROUP BY ALL
),
st AS (
  SELECT s.*, m.gateway_system, m.gateway_reference
  FROM clean_settlement_lines s
  LEFT JOIN map m ON m.cutoff_id = s.cutoff_id AND m.from_system = s.provider_system
                 AND m.from_reference = s.provider_reference
),
settled AS (
  SELECT cutoff_id, gateway_system, gateway_reference, currency_cd, sum(signed_amount_minor) AS settled_minor
  FROM st WHERE gateway_reference IS NOT NULL GROUP BY ALL
),
amounts AS (
  SELECT cutoff_id, gateway_system, gateway_reference, currency_cd,
         coalesce(e.expected_minor, 0) AS expected_minor, coalesce(s.settled_minor, 0) AS settled_minor
  FROM expected e FULL JOIN settled s USING (cutoff_id, gateway_system, gateway_reference, currency_cd)
),
dq AS (
  SELECT DISTINCT d.cutoff_id, m.gateway_system, m.gateway_reference
  FROM dq_money_rows d
  JOIN map m ON m.cutoff_id = d.cutoff_id AND m.from_system = d.from_system AND m.from_reference = d.from_reference
),
linked AS (
  SELECT p.cutoff_id, p.cutoff_ts, p.payment_day, p.payment_key, p.bank_id,
         coalesce(a.currency_cd, p.requested_currency) AS currency_cd,
         coalesce(a.expected_minor, 0) AS expected_minor, coalesce(a.settled_minor, 0) AS settled_minor,
         d.settlement_due_at, q.gateway_reference IS NOT NULL AS dq_flag
  FROM pay p
  LEFT JOIN amounts a ON a.cutoff_id = p.cutoff_id AND a.gateway_system = p.gateway_system
                     AND a.gateway_reference = p.gateway_reference
  LEFT JOIN due d ON d.cutoff_id = p.cutoff_id AND d.gateway_system = p.gateway_system
                 AND d.gateway_reference = p.gateway_reference
  LEFT JOIN dq q ON q.cutoff_id = p.cutoff_id AND q.gateway_system = p.gateway_system
                AND q.gateway_reference = p.gateway_reference
)
SELECT cutoff_id, payment_day, payment_key, bank_id, currency_cd,
       expected_minor, settled_minor, expected_minor - settled_minor AS outstanding_minor,
       CASE WHEN expected_minor = 0 AND settled_minor = 0 THEN 'NOTHING_EXPECTED'
            WHEN expected_minor - settled_minor < 0 THEN 'OVERSETTLED'
            WHEN expected_minor - settled_minor = 0 THEN 'MATCHED'
            WHEN cutoff_ts < settlement_due_at THEN 'PENDING'
            ELSE 'OVERDUE' END AS status,
       settlement_due_at, dq_flag
FROM linked
UNION ALL
SELECT cutoff_id, booking_date AS payment_day, concat_ws('|', provider_system, provider_reference),
       coalesce(bank_id, 'UNKNOWN'), currency_cd,
       0, sum(signed_amount_minor), -sum(signed_amount_minor), 'UNLINKED', CAST(NULL AS TIMESTAMP), false
FROM st
WHERE gateway_reference IS NULL AND booking_date BETWEEN DATE'{window_start}' AND DATE'{day}'
GROUP BY cutoff_id, booking_date, provider_system, provider_reference, bank_id, currency_cd
""")

daily = spark.sql(f"SELECT * FROM q3_window WHERE payment_day = DATE'{day}'")
q2 = spark.table("q2_payment_close").select(*daily.columns)
assert {r.cutoff_id for r in daily.select("cutoff_id").distinct().collect()} == {"C1", "C2"}
assert not daily.exceptAll(q2).take(1) and not q2.exceptAll(daily).take(1), (
    "Notebook 04 must publish both cutoffs for this payment day and release."
)


## Collect source evidence per payment and per unresolved source record

Current gateway, terminal, mapping, processor, refund and settlement records form the
payment evidence. Rejected source versions are eligible for escalation when no current
same-or-higher version replaces them. Conflicting current versions retain all affected
rows. Use a source-scoped key for each case that lacks a usable payment link.


In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW q3_mapping AS
SELECT DISTINCT cutoff_id, from_system, from_reference, gateway_system, gateway_reference,
       concat_ws('|', gateway_system, gateway_reference) AS payment_key
FROM clean_reference_mappings
""")

# Evidence contains source, source row and delivery identifiers.
money_sources = {
    "processor_events": ("processor_system", "processor_reference"),
    "refund_events": ("processor_system", "processor_reference"),
    "settlement_lines": ("provider_system", "provider_reference"),
}
payment_sources = [
    """SELECT g.cutoff_id, g.business_key AS payment_key, 'gateway_requests' AS source_table,
              g.source_row_id, g.delivery_id FROM clean_gateway_requests g""",
    """SELECT g.cutoff_id, g.business_key, 'terminal_assignments', t.source_row_id, t.delivery_id
       FROM clean_gateway_requests g JOIN clean_terminal_assignments t
         ON g.cutoff_id = t.cutoff_id AND g.terminal_system = t.terminal_system
        AND g.terminal_id = t.terminal_id
        AND g.request_timestamp >= t.valid_from AND g.request_timestamp < t.valid_to""",
    """SELECT cutoff_id, concat_ws('|', gateway_system, gateway_reference), 'reference_mappings',
              source_row_id, delivery_id FROM clean_reference_mappings""",
]
for source, (system_col, reference_col) in money_sources.items():
    payment_sources.append(f"""
        SELECT s.cutoff_id, m.payment_key, '{source}', s.source_row_id, s.delivery_id
        FROM clean_{source} s JOIN q3_mapping m
          ON s.cutoff_id = m.cutoff_id AND s.{system_col} = m.from_system
         AND s.{reference_col} = m.from_reference
    """)

spark.sql("""
CREATE OR REPLACE TEMP VIEW q3_payment_evidence AS
SELECT cutoff_id, payment_key,
       sort_array(collect_set(concat_ws('/', source_table, source_row_id, delivery_id))) AS source_refs
FROM (""" + "\nUNION ALL\n".join(payment_sources) + """) AS sources
GROUP BY cutoff_id, payment_key
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW q3_unlinked_evidence AS
SELECT s.cutoff_id, s.booking_date AS payment_day, s.currency_cd,
       coalesce(s.bank_id, 'UNKNOWN') AS bank_id,
       concat_ws('|', s.provider_system, s.provider_reference) AS payment_key,
       sort_array(collect_set(concat_ws('/', 'settlement_lines', s.source_row_id, s.delivery_id))) AS source_refs
FROM clean_settlement_lines s
LEFT ANTI JOIN q3_mapping m
  ON s.cutoff_id = m.cutoff_id AND s.provider_system = m.from_system
 AND s.provider_reference = m.from_reference
GROUP BY ALL
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW q3_states AS
SELECT w.*, coalesce(p.source_refs, u.source_refs, array()) AS source_refs
FROM q3_window w
LEFT JOIN q3_payment_evidence p
  ON w.status <> 'UNLINKED' AND p.cutoff_id = w.cutoff_id AND p.payment_key = w.payment_key
LEFT JOIN q3_unlinked_evidence u
  ON w.status = 'UNLINKED' AND u.cutoff_id = w.cutoff_id AND u.payment_day = w.payment_day
 AND u.payment_key = w.payment_key AND u.bank_id = w.bank_id AND u.currency_cd = w.currency_cd
""")


In [0]:
# Resolve money evidence with a mapping at the same cutoff. A provider-only record
# can still be dated and assigned using its own booking fields.
attribution = {
    "processor_events": ("m.payment_key", "CAST(NULL AS DATE)", "CAST(NULL AS STRING)", "s.currency_cd"),
    "refund_events": ("m.payment_key", "CAST(NULL AS DATE)", "CAST(NULL AS STRING)", "s.currency_cd"),
    "settlement_lines": ("m.payment_key", "try_cast(s.booking_date AS DATE)", "s.bank_id", "s.currency_cd"),
    "gateway_requests": ("s.business_key", "to_date(try_cast(s.request_timestamp AS TIMESTAMP))", "CAST(NULL AS STRING)", "s.currency_cd"),
    "reference_mappings": ("concat_ws('|', s.gateway_system, s.gateway_reference)", "CAST(NULL AS DATE)", "CAST(NULL AS STRING)", "CAST(NULL AS STRING)"),
    "terminal_assignments": ("CAST(NULL AS STRING)", "CAST(NULL AS DATE)", "s.acquiring_bank_id", "CAST(NULL AS STRING)"),
}
quality_selects = []
for source, (payment_key, source_day, bank, currency) in attribution.items():
    mapping_join = ""
    if source in money_sources:
        system_col, reference_col = money_sources[source]
        mapping_join = f"""LEFT JOIN q3_mapping m
          ON m.cutoff_id = s.cutoff_id AND m.from_system = s.{system_col}
         AND m.from_reference = s.{reference_col}"""
    quality_selects.append(f"""
        SELECT s.cutoff_id, '{source}' AS source_table, s.business_key AS source_key,
               s.source_row_id, s.delivery_id, {payment_key} AS payment_key,
               {source_day} AS source_day, {bank} AS source_bank, {currency} AS source_currency,
               CASE WHEN s.row_status = 'QUARANTINED' THEN 'DQ_VERSION_CONFLICT'
                    ELSE concat('DQ_', upper(coalesce(s.validation_issue, 'REJECTED'))) END AS reason
        FROM snap_{source} s
        LEFT ANTI JOIN snap_{source} c
          ON s.row_status = 'REJECTED' AND c.row_status = 'CURRENT'
         AND s.cutoff_id = c.cutoff_id AND s.business_key = c.business_key
         AND c.version_no >= s.version_no
        {mapping_join}
        WHERE s.row_status IN ('REJECTED', 'QUARANTINED')
    """)
spark.sql("CREATE OR REPLACE TEMP VIEW q3_quality_rows AS\n" + "\nUNION ALL\n".join(quality_selects))

# Attribute against the full gateway history so cases known to be outside this
# investigation window are distinguished from cases whose day is unknown.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW q3_quality_cases AS
WITH attributed AS (
    SELECT d.cutoff_id,
           coalesce(to_date(g.request_timestamp), d.source_day) AS payment_day,
           coalesce(w.bank_id, t.acquiring_bank_id, d.source_bank, 'UNKNOWN') AS bank_id,
           coalesce(w.currency_cd, d.source_currency, g.currency_cd, 'UNKNOWN') AS currency_cd,
           CASE WHEN g.business_key IS NOT NULL THEN 'PAYMENT' ELSE 'SOURCE' END AS affected_type,
           coalesce(g.business_key, concat_ws('/', d.source_table, coalesce(d.source_key, d.source_row_id))) AS affected_key,
           d.reason, d.source_table, d.source_row_id, d.delivery_id
    FROM q3_quality_rows d
    LEFT JOIN clean_gateway_requests g
      ON g.cutoff_id = d.cutoff_id AND g.business_key = d.payment_key
    LEFT JOIN clean_terminal_assignments t
      ON t.cutoff_id = g.cutoff_id AND t.terminal_system = g.terminal_system
     AND t.terminal_id = g.terminal_id
     AND g.request_timestamp >= t.valid_from AND g.request_timestamp < t.valid_to
    LEFT JOIN q3_window w
      ON w.cutoff_id = g.cutoff_id AND w.payment_key = g.business_key AND w.status <> 'UNLINKED'
     AND w.currency_cd = coalesce(d.source_currency, g.currency_cd)
)
SELECT cutoff_id, payment_day, bank_id, currency_cd, affected_type, affected_key, reason,
       'Review the source evidence and correct or confirm the affected records' AS action,
       true AS is_blocking,
       sort_array(collect_set(concat_ws('/', source_table, source_row_id, delivery_id))) AS source_refs
FROM attributed
WHERE payment_day BETWEEN DATE'{window_start}' AND DATE'{day}' OR payment_day IS NULL
GROUP BY ALL
""")


## Publish the exception queue, daily close and C2 investigation

The queue covers the seven-day investigation and retains undated exceptions. The
daily close uses D only. The C2 investigation includes monitoring cases and blockers
with dates in the window. Finance also receives an undated blocking-case count by cutoff.


In [0]:
publish("q3_exception_queue", """
SELECT cutoff_id, payment_day, bank_id, currency_cd,
       CASE WHEN status = 'UNLINKED' THEN 'PROVIDER_REFERENCE' ELSE 'PAYMENT' END AS affected_type,
       payment_key AS affected_key, status AS reason,
       CASE status
           WHEN 'PENDING' THEN 'Monitor until the settlement due time'
           WHEN 'OVERDUE' THEN 'Investigate the missing settlement with the provider'
           WHEN 'OVERSETTLED' THEN 'Verify excess postings and request a correction'
           WHEN 'UNLINKED' THEN 'Identify the original payment and repair the reference mapping'
       END AS action,
       status <> 'PENDING' AS is_blocking, source_refs
FROM q3_states
WHERE status IN ('PENDING', 'OVERDUE', 'OVERSETTLED', 'UNLINKED')
UNION ALL
SELECT * FROM q3_quality_cases
""")

publish("q3_daily_close", f"""
WITH totals AS (
    SELECT cutoff_id, payment_day, bank_id, currency_cd,
           sum(expected_minor) AS expected_minor, sum(settled_minor) AS settled_minor,
           sum(outstanding_minor) AS outstanding_minor,
           count_if(status = 'MATCHED') AS matched_count,
           count_if(status = 'PENDING') AS pending_count,
           count_if(status = 'OVERDUE') AS overdue_count,
           count_if(status = 'OVERSETTLED') AS oversettled_count,
           count_if(status = 'UNLINKED') AS unlinked_count,
           count_if(status = 'NOTHING_EXPECTED') AS nothing_expected_count
    FROM q3_window WHERE payment_day = DATE'{day}' GROUP BY ALL
), blockers AS (
    SELECT cutoff_id, payment_day, bank_id, currency_cd, count(*) AS blocking_case_count
    FROM q3_exception_queue
    WHERE payment_day = DATE'{day}' AND is_blocking GROUP BY ALL
)
SELECT cutoff_id, payment_day, bank_id, currency_cd,
       coalesce(expected_minor, 0) AS expected_minor,
       coalesce(settled_minor, 0) AS settled_minor,
       coalesce(outstanding_minor, 0) AS outstanding_minor,
       coalesce(matched_count, 0) AS matched_count, coalesce(pending_count, 0) AS pending_count,
       coalesce(overdue_count, 0) AS overdue_count, coalesce(oversettled_count, 0) AS oversettled_count,
       coalesce(unlinked_count, 0) AS unlinked_count,
       coalesce(nothing_expected_count, 0) AS nothing_expected_count,
       coalesce(blocking_case_count, 0) AS blocking_case_count,
       CASE WHEN blocking_case_count > 0 THEN 'HOLD' ELSE 'SIGN_OFF' END AS sign_off_decision
FROM totals FULL OUTER JOIN blockers USING (cutoff_id, payment_day, bank_id, currency_cd)
""")

publish("q3_c2_investigation", f"""
SELECT * FROM q3_exception_queue
WHERE cutoff_id = 'C2' AND payment_day BETWEEN DATE'{window_start}' AND DATE'{day}'
""")

display(spark.sql("SELECT * FROM q3_daily_close ORDER BY bank_id, currency_cd, cutoff_id"))
display(spark.sql("""
SELECT cutoff_id, bank_id, currency_cd, reason, count(*) AS undated_blockers
FROM q3_exception_queue WHERE is_blocking AND payment_day IS NULL GROUP BY ALL
ORDER BY cutoff_id, bank_id, currency_cd, reason
"""))
display(spark.sql("""
SELECT payment_day, bank_id, currency_cd, reason, count(*) AS case_count
FROM q3_c2_investigation GROUP BY ALL ORDER BY payment_day, bank_id, currency_cd, reason
"""))


## Explain changes from C1 to C2

The payment comparison carries expected and settled amounts, status, and source rows
added or removed from the current evidence. A PENDING-to-OVERDUE move with unchanged
amounts can occur when the C2 cutoff reaches the due time. An unlinked reference
resolved by C2 carries the C2 mapping rows and target payment keys. The queue comparison
also shows cases that opened, cleared or changed evidence between cutoffs.


In [0]:
publish("q3_payment_changes", f"""
WITH c1 AS (
    SELECT * FROM q3_states WHERE cutoff_id = 'C1' AND payment_day = DATE'{day}'
), c2 AS (
    SELECT * FROM q3_states WHERE cutoff_id = 'C2' AND payment_day = DATE'{day}'
), c2_mappings AS (
    SELECT concat_ws('|', from_system, from_reference) AS provider_key,
           sort_array(collect_set(concat_ws('|', gateway_system, gateway_reference))) AS mapped_payments,
           sort_array(collect_set(concat_ws('/', 'reference_mappings', source_row_id, delivery_id))) AS mapping_refs
    FROM clean_reference_mappings WHERE cutoff_id = 'C2' GROUP BY ALL
)
SELECT coalesce(c1.payment_key, c2.payment_key) AS affected_key,
       c1.bank_id AS bank_c1, c2.bank_id AS bank_c2,
       coalesce(c1.currency_cd, c2.currency_cd) AS currency_cd,
       c1.status AS status_c1, c2.status AS status_c2,
       c1.dq_flag AS dq_flag_c1, c2.dq_flag AS dq_flag_c2,
       c1.expected_minor AS expected_c1, c2.expected_minor AS expected_c2,
       c1.settled_minor AS settled_c1, c2.settled_minor AS settled_c2,
       c1.outstanding_minor AS outstanding_c1, c2.outstanding_minor AS outstanding_c2,
       c2.settlement_due_at AS due_c2,
       m.mapped_payments, m.mapping_refs,
       array_except(coalesce(c2.source_refs, array()), coalesce(c1.source_refs, array())) AS added_current_refs,
       array_except(coalesce(c1.source_refs, array()), coalesce(c2.source_refs, array())) AS removed_current_refs,
       CASE
           WHEN c1.status = 'UNLINKED' AND c2.status IS NULL AND m.provider_key IS NOT NULL
               THEN 'Provider reference has a C2 mapping. Check its target payment and booking evidence.'
           WHEN c1.status = 'PENDING' AND c2.status = 'OVERDUE'
            AND c1.expected_minor = c2.expected_minor AND c1.settled_minor = c2.settled_minor
            AND c1.settlement_due_at = c2.settlement_due_at
               THEN 'Settlement due time reached by C2'
           WHEN c1.expected_minor <> c2.expected_minor THEN 'Expected amount changed. Review capture, reversal and refund evidence.'
           WHEN c1.settled_minor <> c2.settled_minor THEN 'Settled amount changed. Review settlement and mapping evidence.'
           ELSE 'Review changes in current source records and the exception queue'
       END AS explanation
FROM c1 FULL OUTER JOIN c2
  ON c1.payment_key = c2.payment_key AND c1.currency_cd = c2.currency_cd
 AND (c1.status = 'UNLINKED') = (c2.status = 'UNLINKED')
 AND (c1.status <> 'UNLINKED' OR c1.bank_id = c2.bank_id)
LEFT JOIN c2_mappings m ON c1.status = 'UNLINKED' AND c1.payment_key = m.provider_key
WHERE NOT (c1.status <=> c2.status)
   OR NOT (c1.bank_id <=> c2.bank_id)
   OR NOT (c1.expected_minor <=> c2.expected_minor)
   OR NOT (c1.settled_minor <=> c2.settled_minor)
   OR NOT (c1.dq_flag <=> c2.dq_flag)
""")

publish("q3_exception_changes", f"""
WITH c1 AS (
    SELECT * FROM q3_exception_queue WHERE cutoff_id = 'C1'
    AND (payment_day = DATE'{day}' OR payment_day IS NULL)
), c2 AS (
    SELECT * FROM q3_exception_queue WHERE cutoff_id = 'C2'
    AND (payment_day = DATE'{day}' OR payment_day IS NULL)
)
SELECT coalesce(c1.payment_day, c2.payment_day) AS payment_day,
       coalesce(c1.bank_id, c2.bank_id) AS bank_id,
       coalesce(c1.currency_cd, c2.currency_cd) AS currency_cd,
       coalesce(c1.affected_type, c2.affected_type) AS affected_type,
       coalesce(c1.affected_key, c2.affected_key) AS affected_key,
       coalesce(c1.reason, c2.reason) AS reason,
       CASE WHEN c1.affected_key IS NULL THEN 'ADDED'
            WHEN c2.affected_key IS NULL THEN 'REMOVED'
            ELSE 'EVIDENCE_CHANGED' END AS change,
       c1.source_refs AS refs_c1, c2.source_refs AS refs_c2
FROM c1 FULL OUTER JOIN c2
  ON c1.payment_day <=> c2.payment_day
 AND c1.bank_id = c2.bank_id AND c1.currency_cd = c2.currency_cd
 AND c1.affected_type = c2.affected_type AND c1.affected_key = c2.affected_key
 AND c1.reason = c2.reason
WHERE NOT (c1.source_refs <=> c2.source_refs)
""")

display(spark.sql("""
SELECT status_c1, status_c2, count(*) AS affected_rows
FROM q3_payment_changes GROUP BY ALL ORDER BY status_c1, status_c2
"""))
display(spark.sql("""
SELECT * FROM q3_payment_changes
WHERE affected_key = 'GW01|000039372'
"""))
display(spark.sql("""
SELECT change, reason, count(*) AS cases FROM q3_exception_changes
GROUP BY ALL ORDER BY change, reason
"""))


In [0]:
assert not spark.sql("""
    SELECT cutoff_id, payment_day, bank_id, currency_cd, affected_type, affected_key, reason
    FROM q3_exception_queue GROUP BY ALL HAVING count(*) > 1
""").take(1), "Duplicate case at the queue grain."
assert not spark.sql("""
    SELECT * FROM q3_exception_queue WHERE size(source_refs) = 0
""").take(1), "Every queue case needs source evidence."
assert not spark.sql("""
    SELECT * FROM q3_daily_close
    WHERE outstanding_minor <> expected_minor - settled_minor
       OR (sign_off_decision = 'HOLD') <> (blocking_case_count > 0)
""").take(1), "Daily close amounts or sign-off counts disagree."
assert not spark.sql("""
    SELECT * FROM q3_daily_close
    WHERE outstanding_minor = 0 AND blocking_case_count > 0 AND sign_off_decision <> 'HOLD'
""").take(1), "A balanced net amount can still have blocking cases."
